# 01 Generate Data

Configuration-driven data generation for autonomous continuous-time systems. The notebook only selects the system and dataset mode; simulation, dataset construction, metadata, and file naming live in `stable_icnn_physics`.

## Configuration

Change `SYSTEM_NAME` or `SYSTEM_KWARGS` here. Later notebooks read `latest_dataset_config.json`, so they automatically use the dataset generated by this run.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd() if (Path.cwd() / "stable_icnn_physics").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

for module_name in list(sys.modules):
    if module_name == "stable_icnn_physics" or module_name.startswith("stable_icnn_physics."):
        del sys.modules[module_name]

import json
import numpy as np
import matplotlib.pyplot as plt

from stable_icnn_physics import make_system, available_systems
from stable_icnn_physics.data import (
    dataset_base_name,
    dataset_path,
    generate_derivative_data,
    save_dataset,
    save_trajectory_dataset,
    simulate_trajectories,
)

SYSTEM_NAME = "damped_pendulum_4"
DATASET_MODE = "trajectory_derivative"  # random_derivative, trajectory_derivative, trajectory_transition
CACHE_DIR = REPO_ROOT / "data" / "cache"

N_TRAIN_TRAJ = 200
N_TEST_TRAJ = 50
STEPS = 1000
DT = 0.02

N_RANDOM_TRAIN = 50000
N_RANDOM_TEST = 10000
SEED = 0

SYSTEM_KWARGS = {
    "friction": 0.3,
    "gravity": 9.81,
    "angle_range": (-np.pi, np.pi),
    "velocity_range": (-np.pi, np.pi),
}

# For custom systems, define a callable and switch SYSTEM_NAME to "custom_state_space".
# def my_rhs(x, params):
#     return np.stack([x[:, 1], -params["k"] * x[:, 0]], axis=1)
# SYSTEM_KWARGS = {
#     "name": "my_system",
#     "state_dim": 2,
#     "rhs_fn": my_rhs,
#     "state_ranges": [(-2, 2), (-2, 2)],
#     "angle_indices": [],
#     "params": {"k": 1.0},
# }

print("Available examples:", available_systems())
system = make_system(SYSTEM_NAME, **SYSTEM_KWARGS)
print(json.dumps(system.metadata(), indent=2))


## Generate And Save

The saved `.npz` files contain `x`, `y`, JSON metadata, and for trajectory modes also the raw `trajectories` and `derivatives` arrays.

In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def base_metadata(split, dataset_type, generated_from, n_trajectories=None, n_samples=None):
    return {
        "system_name": system.name,
        "state_dim": system.state_dim,
        "system": system.metadata(),
        "split": split,
        "seed": SEED,
        "dt": DT if n_trajectories is not None else None,
        "steps": STEPS if n_trajectories is not None else None,
        "n_trajectories": n_trajectories,
        "n_samples": n_samples,
        "dataset_type": dataset_type,
        "generated_from": generated_from,
        "dataset_mode": DATASET_MODE,
    }

generated = {}
trajectory_paths = {}

if DATASET_MODE == "random_derivative":
    for split, n_samples in [("train", N_RANDOM_TRAIN), ("test", N_RANDOM_TEST)]:
        x, y = generate_derivative_data(system, n_samples=n_samples, split=split, seed=SEED)
        path = CACHE_DIR / dataset_base_name(system, split=split, n_samples=n_samples, seed=SEED, dataset_type="derivative")
        save_dataset(path, x, y, metadata=base_metadata(split, "derivative", "random_state_samples", n_samples=n_samples))
        generated[split] = {"x": x, "y": y, "path": path}
elif DATASET_MODE in {"trajectory_derivative", "trajectory_transition"}:
    dataset_type = "transition" if DATASET_MODE == "trajectory_transition" else "derivative"
    for split, n_traj in [("train", N_TRAIN_TRAJ), ("test", N_TEST_TRAJ)]:
        trajectories, derivatives = simulate_trajectories(system, n_trajectories=n_traj, steps=STEPS, dt=DT, split=split, seed=SEED)
        path = CACHE_DIR / dataset_base_name(
            system, split=split, n_trajectories=n_traj, steps=STEPS, dt=DT, seed=SEED, dataset_type=dataset_type
        )
        save_trajectory_dataset(
            path,
            trajectories,
            derivatives,
            metadata=base_metadata(split, dataset_type, "trajectories", n_trajectories=n_traj),
        )
        data = np.load(path, allow_pickle=False)
        generated[split] = {"x": data["x"].astype(np.float32), "y": data["y"].astype(np.float32), "path": path, "trajectories": trajectories}
        trajectory_paths[split] = path
else:
    raise ValueError(f"Unknown DATASET_MODE: {DATASET_MODE}")

config = {
    "train_dataset_path": str(generated["train"]["path"]),
    "test_dataset_path": str(generated["test"]["path"]),
    "train_trajectory_path": str(trajectory_paths.get("train")) if "train" in trajectory_paths else None,
    "test_trajectory_path": str(trajectory_paths.get("test")) if "test" in trajectory_paths else None,
    "system": system.metadata(),
    "dataset_mode": DATASET_MODE,
    "dt": DT,
    "steps": STEPS,
    "seed": SEED,
}
config_path = CACHE_DIR / "latest_dataset_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

for split, item in generated.items():
    print(split, item["path"])
    print("  x/y:", item["x"].shape, item["y"].shape, "finite:", np.isfinite(item["x"]).all() and np.isfinite(item["y"]).all())
print("config:", config_path)


## Sanity Plots

In [ ]:
item = generated["train"]
x = item["x"]

if system.state_dim == 2:
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(x[: min(len(x), 4000), 0], x[: min(len(x), 4000), 1], s=4, alpha=0.25)
    ax.set_xlabel("state[0]")
    ax.set_ylabel("state[1]")
    ax.set_title(f"Sampled states: {system.name}")
    ax.grid(alpha=0.25)
else:
    trajectories = item.get("trajectories")
    if trajectories is None:
        sample = x[: min(len(x), 2000)]
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(sample[:, : min(system.state_dim, 8)])
        axes[0].set_title("Random state samples")
        axes[0].grid(alpha=0.25)
        axes[1].hist(sample.ravel(), bins=60)
        axes[1].set_title("State value histogram")
        axes[1].grid(alpha=0.25)
    else:
        traj = trajectories[0]
        time = np.arange(traj.shape[0]) * DT
        n_show = min(system.state_dim, 8)
        fig, ax = plt.subplots(figsize=(12, 5))
        for j in range(n_show):
            label = f"theta[{j}]" if j < system.state_dim // 2 else f"omega[{j - system.state_dim // 2}]"
            ax.plot(time, traj[:, j], label=label)
        ax.set_xlabel("time")
        ax.set_title(f"One trajectory: {system.name}")
        ax.legend(ncol=2)
        ax.grid(alpha=0.25)
plt.tight_layout()
